# 07 · Dataclasses, Enums, NamedTuples & `__slots__` 🆕

**Module 06 – Object-Oriented Programming** · Estimated time: 55 min

> ❗ Missing from both sources (the bootcamp only shows `collections.namedtuple`). These are everyday tools in modern Python.

### 🎯 What you will learn
- `@dataclass`: classes for data, without the boilerplate
- Defaults, `field(default_factory=...)`, `__post_init__`, `frozen`, `order`, `slots`, `kw_only`
- `asdict`, `astuple`, `replace` / 🆕 `copy.replace()` (3.13)
- `typing.NamedTuple` vs `collections.namedtuple`
- `enum`: `Enum`, `auto`, `IntEnum`, 🆕 `StrEnum` (3.11), `Flag`
- `__slots__`: memory-efficient objects
- ⚠️ 3.15 removal: keyword-argument syntax for `NamedTuple`

## 1. The boilerplate problem

In [1]:
class PointOld:
    def __init__(self, x, y):
        self.x = x
        self.y = y
    def __repr__(self):
        return f"PointOld(x={self.x}, y={self.y})"
    def __eq__(self, other):
        return (self.x, self.y) == (other.x, other.y)

print(PointOld(1, 2), PointOld(1, 2) == PointOld(1, 2))

PointOld(x=1, y=2) True


## 2. ✅ `@dataclass` (3.7+) writes `__init__`, `__repr__`, `__eq__` for you

In [2]:
from dataclasses import dataclass, field, asdict, astuple, replace

@dataclass
class Point:
    x: float
    y: float

p = Point(1, 2)
print(p, p == Point(1, 2), p.x)

Point(x=1, y=2) True 1


The type hints define the **fields** (they're required for dataclasses, but not enforced at runtime).

### Defaults and mutable defaults

In [3]:
@dataclass
class Student:
    name: str
    age: int = 18
    courses: list[str] = field(default_factory=list)   # ✅ new list per instance
    # courses: list[str] = []  ← ❌ raises ValueError: mutable default not allowed

s1 = Student("Asha")
s2 = Student("Ravi", 21, ["Python"])
s1.courses.append("Math")
print(s1, s2, sep="\n")

Student(name='Asha', age=18, courses=['Math'])
Student(name='Ravi', age=21, courses=['Python'])


### `__post_init__`: validation & derived fields

In [4]:
@dataclass
class Rectangle:
    width: float
    height: float
    area: float = field(init=False)          # computed, not passed in

    def __post_init__(self):
        if self.width <= 0 or self.height <= 0:
            raise ValueError("dimensions must be positive")
        self.area = self.width * self.height

print(Rectangle(3, 4))
try:
    Rectangle(-1, 2)
except ValueError as e:
    print("ValueError:", e)

Rectangle(width=3, height=4, area=12)
ValueError: dimensions must be positive


### Options: `frozen`, `order`, `slots`, `kw_only`

In [5]:
@dataclass(frozen=True, order=True, slots=True)       # slots: 3.10+
class Version:
    major: int
    minor: int
    patch: int = 0

v = Version(3, 15)
print(sorted([Version(3, 14, 2), v, Version(3, 9)]))   # order=True → comparable
print({v: "latest"})                                    # frozen → hashable
try:
    v.major = 4
except Exception as e:
    print(type(e).__name__, e)

[Version(major=3, minor=9, patch=0), Version(major=3, minor=14, patch=2), Version(major=3, minor=15, patch=0)]
{Version(major=3, minor=15, patch=0): 'latest'}
FrozenInstanceError cannot assign to field 'major'


In [6]:
@dataclass(kw_only=True)                     # 3.10+: all fields keyword-only
class Config:
    host: str = "localhost"
    port: int = 8080
    debug: bool = False

print(Config(port=9000, debug=True))

Config(host='localhost', port=9000, debug=True)


### Converting & copying

In [7]:
import copy
p = Point(1, 2)
print(asdict(p), astuple(p))
p2 = replace(p, y=99)            # a new object with one field changed
p3 = copy.replace(p, x=-1)       # 🆕 3.13: generic copy.replace() works for dataclasses, namedtuples, datetime, ...
print(p, p2, p3)

{'x': 1, 'y': 2} (1, 2)
Point(x=1, y=2) Point(x=1, y=99) Point(x=-1, y=2)


### Inheritance with dataclasses

In [8]:
@dataclass
class Animal:
    name: str
    sound: str = "..."
    def speak(self):
        return f"{self.name} says {self.sound}"

@dataclass
class Dog(Animal):
    sound: str = "Woof"
    tricks: list[str] = field(default_factory=list)

print(Dog("Rex", tricks=["sit"]), Dog("Rex").speak())

Dog(name='Rex', sound='Woof', tricks=['sit']) Rex says Woof


## 3. Named tuples

Immutable, lightweight records that also behave like tuples (indexing, unpacking).

### `collections.namedtuple` (📗 Collections module)

In [9]:
from collections import namedtuple

Dog = namedtuple("Dog", ["age", "breed", "name"])
sam = Dog(age=2, breed="Lab", name="Sammy")
print(sam, sam.age, sam[1])
age, breed, name = sam
print(sam._asdict(), sam._replace(age=3))

Dog(age=2, breed='Lab', name='Sammy') 2 Lab
{'age': 2, 'breed': 'Lab', 'name': 'Sammy'} Dog(age=3, breed='Lab', name='Sammy')


### ✅ `typing.NamedTuple` (modern, with type hints and methods)

In [10]:
from typing import NamedTuple

class Coordinate(NamedTuple):
    lat: float
    lon: float
    label: str = "unknown"

    def to_str(self) -> str:
        return f"{self.label}: {self.lat:.3f}, {self.lon:.3f}"

pune = Coordinate(18.5204, 73.8567, "Pune")
print(pune, pune.to_str(), pune[0])

Coordinate(lat=18.5204, lon=73.8567, label='Pune') Pune: 18.520, 73.857 18.5204


> ⚠️ **Removed in Python 3.15:** the keyword-argument form `Point = NamedTuple("Point", x=int, y=int)`. Use the class syntax above, or the functional form with a list: `NamedTuple("Point", [("x", int), ("y", int)])`.

In [11]:
Point2 = NamedTuple("Point2", [("x", int), ("y", int)])    # ✅ still valid
print(Point2(1, 2))

Point2(x=1, y=2)


### dataclass vs NamedTuple vs dict

| | `dict` | `NamedTuple` | `@dataclass` |
|---|---|---|---|
| Field access | `d["x"]` | `p.x` | `p.x` |
| Mutable | ✅ | ❌ | ✅ (or `frozen=True`) |
| Tuple behaviour (unpack/index) | ❌ | ✅ | ❌ |
| Methods / validation | ❌ | limited | ✅ `__post_init__` |
| Typo protection | ❌ | ✅ | ✅ |
| Best for | dynamic/JSON data | small immutable records | most custom data classes |

## 4. Enumerations: `enum`

An **Enum** is a set of named constant values. No more "magic strings" like `"pendng"` typos.

In [12]:
from enum import Enum, auto, IntEnum, StrEnum, Flag

class Status(Enum):
    PENDING = 1
    SHIPPED = 2
    DELIVERED = 3

s = Status.SHIPPED
print(s, s.name, s.value, repr(s))
print(Status(3), Status["PENDING"])            # look up by value / by name
print(s is Status.SHIPPED, s == 2)             # Enum members are NOT equal to raw values
print([m.name for m in Status], len(Status))

Status.SHIPPED SHIPPED 2 <Status.SHIPPED: 2>
Status.DELIVERED Status.PENDING
True False
['PENDING', 'SHIPPED', 'DELIVERED'] 3


In [13]:
class Color(Enum):
    RED = auto()          # auto-assigned values
    GREEN = auto()
    BLUE = auto()

    def describe(self):
        return f"{self.name.title()} is colour #{self.value}"

print(list(Color), Color.GREEN.describe())

[<Color.RED: 1>, <Color.GREEN: 2>, <Color.BLUE: 3>] Green is colour #2


### Specialised enums

In [14]:
class Priority(IntEnum):           # behaves like an int
    LOW = 1
    HIGH = 3
print(Priority.HIGH > Priority.LOW, Priority.HIGH + 1, sorted([Priority.HIGH, Priority.LOW]))

class Method(StrEnum):             # 🆕 3.11: behaves like a str
    GET = auto()                   # auto() → lowercase name: "get"
    POST = auto()
print(Method.GET, Method.GET == "get", Method.POST.upper())

class Permission(Flag):            # combinable bit flags
    READ = auto()
    WRITE = auto()
    EXECUTE = auto()
perms = Permission.READ | Permission.WRITE
print(perms, Permission.WRITE in perms, Permission.EXECUTE in perms)

True 4 [<Priority.LOW: 1>, <Priority.HIGH: 3>]
get True POST
Permission.READ|WRITE True False


### Enums + `match` (from Module 04)

In [15]:
def next_step(status: Status) -> str:
    match status:
        case Status.PENDING:
            return "Pack the order"
        case Status.SHIPPED:
            return "Track the parcel"
        case Status.DELIVERED:
            return "Ask for a review"

print(next_step(Status.SHIPPED))

Track the parcel


## 5. `__slots__`: fixed attributes, less memory

Normal objects store attributes in a per-instance `__dict__`. With `__slots__`, Python uses a compact fixed layout: less memory and slightly faster access, but you can't add new attributes.

In [16]:
import sys

class RegularPoint:
    def __init__(self, x, y):
        self.x, self.y = x, y

class SlotPoint:
    __slots__ = ("x", "y")
    def __init__(self, x, y):
        self.x, self.y = x, y

r, s = RegularPoint(1, 2), SlotPoint(1, 2)
print(sys.getsizeof(r) + sys.getsizeof(r.__dict__), "bytes vs", sys.getsizeof(s), "bytes")
r.z = 3                     # fine
try:
    s.z = 3
except AttributeError as e:
    print("AttributeError:", e)

344 bytes vs 48 bytes
AttributeError: 'SlotPoint' object has no attribute 'z' and no __dict__ for setting new attributes


> 💡 With dataclasses, just use `@dataclass(slots=True)`.

> ➡️ Dataclasses don't validate data. For validated models with JSON parsing and serialization, see **Module 15 · Pydantic**. How `@dataclass`, `property` and `__slots__` work internally: **Module 10 · 02–03**.

## 📝 Summary
- `@dataclass` auto-generates `__init__`, `__repr__`, `__eq__`; use `field(default_factory=...)` for mutable defaults.
- `frozen=True` → immutable & hashable; `order=True` → sortable; `slots=True` → memory-efficient.
- `NamedTuple` for small immutable records that are also tuples.
- `Enum`/`IntEnum`/`StrEnum`/`Flag` replace magic constants.

## 🏋️ Practice
1. Create a frozen dataclass `Money(amount: Decimal, currency: str)` with a `__post_init__` that rejects negative amounts.
2. Create a dataclass `Order` with `items: list[str]` (default empty), `status: Status = Status.PENDING`, and a method `ship()`.
3. Create a `Weekday` `IntEnum` (MON=1..SUN=7) and a function `is_weekend(day)`.
4. Convert `Order` to a dict with `asdict`.

<details><summary>Show solutions</summary>

```python
from dataclasses import dataclass, field, asdict
from decimal import Decimal
from enum import Enum, IntEnum

class Status(Enum):              # same as in the lesson
    PENDING = 1
    SHIPPED = 2
    DELIVERED = 3

@dataclass(frozen=True)
class Money:
    amount: Decimal
    currency: str = "INR"
    def __post_init__(self):
        if self.amount < 0:
            raise ValueError("negative amount")

@dataclass
class Order:
    items: list[str] = field(default_factory=list)
    status: Status = Status.PENDING
    def ship(self):
        self.status = Status.SHIPPED

class Weekday(IntEnum):
    MON = 1; TUE = 2; WED = 3; THU = 4; FRI = 5; SAT = 6; SUN = 7
def is_weekend(day: Weekday) -> bool:
    return day >= Weekday.SAT

print(asdict(Order(["pen"])))
```
</details>